# PixelRAG — Screenshot Embedding Pipeline

This notebook demonstrates end-to-end visual RAG:

1. **Capture** — use `fossick` to take full-page PNG screenshots via Chrome CDP
2. **Tile** — split tall screenshots into overlapping tiles with `tile_image`
3. **Embed** — encode tiles with `FastEncodeMultimodal` (SigLIP2, single model, shared text/image space)
4. **Index** — store embeddings in litesearch's SQLite image store
5. **Search** — query with plain text; retrieve matching screenshot tiles
6. **Read** *(optional)* — pass retrieved tiles to `granite-docling-258M` for structured extraction

SigLIP2 maps both text and images into the same vector space, so a text query directly retrieves
the most visually relevant screenshot tile — no separate text-embedding model needed at inference time.

In [ ]:
# pip install fossick litesearch pillow
# For the optional reader step: pip install transformers accelerate

## 1. Capture screenshots with fossick

In [ ]:
from fossick import automation_browser
from pathlib import Path

SCREENSHOT_DIR = Path('./screenshots')
SCREENSHOT_DIR.mkdir(exist_ok=True)

URLS = [
    'https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)',
    'https://en.wikipedia.org/wiki/Retrieval-augmented_generation',
]

screenshots = []  # list of (url, path)

with automation_browser() as s:
    for url in URLS:
        slug = url.split('/')[-1][:40]
        out = str(SCREENSHOT_DIR / f'{slug}.png')
        path = s.screenshot(url=url, path=out, full_page=True)
        screenshots.append((url, path))
        print(f'captured {path}')

print(f'\nTotal pages: {len(screenshots)}')

## 2. Tile screenshots

`tile_image` splits a tall full-page capture into overlapping 1024-px strips.
Each tile becomes an independently searchable unit.

In [ ]:
from litesearch import tile_image

TILE_DIR = Path('./tiles')
TILE_DIR.mkdir(exist_ok=True)

tiles = []  # list of (url, tile_index, tile_path)

for url, img_path in screenshots:
    for i, tile in enumerate(tile_image(img_path, tile_h=1024, overlap=128)):
        tile_path = str(TILE_DIR / f'{Path(img_path).stem}_tile{i:03d}.png')
        tile.save(tile_path)
        tiles.append((url, i, tile_path))

print(f'Total tiles: {len(tiles)}')

## 3. Embed tiles with SigLIP2

`FastEncodeMultimodal` wraps both the vision and text SigLIP2 encoders.
A single model instance handles both image embedding (index time) and text embedding (query time).

Use `siglip2_base` (256px, ~375M params) for CPU deployments, or `siglip2_so400m` (512px, ~880M) for best quality.

In [ ]:
from litesearch import FastEncodeMultimodal, siglip2_base

# Download ~375 MB on first run; cached locally afterwards
enc = FastEncodeMultimodal(model_dict=siglip2_base)

tile_paths = [t for _, _, t in tiles]
embeddings = enc.encode_image(tile_paths)  # shape (N, 768), dtype float16

print(f'Embedded {len(embeddings)} tiles, dim={embeddings.shape[1]}, dtype={embeddings.dtype}')

## 4. Index into litesearch

In [ ]:
from litesearch import database
import numpy as np

db = database('pixelrag.db')
store = db.get_image_store()  # creates image_store table with FTS5 + embedding column

rows = [
    dict(
        url=url,
        tile_path=tile_path,
        tile_index=tile_idx,
        embedding=emb.ravel().tobytes(),
        page_text=url,          # store URL as searchable text; add page markdown here if available
    )
    for (url, tile_idx, tile_path), emb in zip(tiles, embeddings)
]

store.insert_all(rows)
print(f'Indexed {store.count} tiles into pixelrag.db')

## 5. Search

At query time only `enc.encode_text()` is called — same model, no extra download.

In [ ]:
QUERY = 'attention mechanism diagram'

q_emb = enc.encode_text([QUERY])[0]                # (768,) float16
results = store.vec_search(
    emb=q_emb.ravel().tobytes(),
    columns=['id', 'url', 'tile_path', 'tile_index'],
    dtype=np.float16,
    limit=5,
)

for r in results:
    print(f"[dist={r['_dist']:.4f}] tile {r['tile_index']:>3}  {r['tile_path']}")

In [ ]:
# Display top result inline (Jupyter)
from IPython.display import Image as IpyImage, display
if results:
    display(IpyImage(filename=results[0]['tile_path'], width=700))

## 6. (Optional) Read retrieved tiles with granite-docling-258M

`granite-docling-258M` (IBM, Apache-2.0, 258M params) reads a screenshot and returns structured
markdown — preserving tables, code blocks, and formulas that OCR would mangle.
Use it as the reader stage after retrieval.

In [ ]:
#| eval: false
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image

reader_id = 'ibm-granite/granite-docling-258M'
processor = AutoProcessor.from_pretrained(reader_id)
reader = AutoModelForVision2Seq.from_pretrained(reader_id)
reader.eval()

def read_tile(tile_path: str) -> str:
    """Run granite-docling on a tile PNG; returns extracted markdown."""
    img = Image.open(tile_path).convert('RGB')
    inputs = processor(images=img, text='Convert this page to markdown.', return_tensors='pt')
    ids = reader.generate(**inputs, max_new_tokens=1024)
    return processor.decode(ids[0], skip_special_tokens=True)

if results:
    text = read_tile(results[0]['tile_path'])
    print(text[:2000])

## Pipeline summary

```
fossick.screenshot(url)          → full-page PNG
    └─ tile_image(png)           → list of 1024px tile PNGs
         └─ enc.encode_image()   → float16 embeddings  (SigLIP2-base, single model)
              └─ image_store     → SQLite + usearch SIMD (litesearch)

query (text)
    └─ enc.encode_text()         → float16 query vector (same SigLIP2 model)
         └─ store.vec_search()   → top-k tiles  [+ FTS fallback via db.search()]
              └─ granite-docling → structured markdown from tile image  (optional)
```

To scale beyond ~200K tiles, swap `store.vec_search()` for a usearch HNSW index
keyed by `id` with metadata living in SQLite.

For higher retrieval quality on screenshot-heavy corpora, swap `siglip2_base` for
`Qwen/Qwen3-VL-Embedding-2B` (native Matryoshka, screenshot-tuned) — requires a
PyTorch-based encoder wrapper since no ONNX is available yet.